In [ ]:
import pandas as pd
import numpy as np

from combat.pycombat import pycombat  # pip install combat

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import *
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import umap
import seaborn as sns  # For h-clust mostly
import plotly.express as px
import plotly.graph_objects as go  # For heatmaps

In [ ]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

## Functions

In [ ]:
def plot_df(a_df, episign_of_interest):
    # Plot df resulting from t-SNE or Isomap
    color_selected = [x in episign_of_interest for x in a_df.index]
    fig = px.scatter(
            a_df,
            x='compon0',
            y='compon1',
            hover_data=[a_df.index],
            color=color_selected
    )
    return fig

# MAIN

In [ ]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("counts_norm.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
# MEMO: Cannot compute epiSize with 'full values' (almost NO '0' values)
#epiSize = {}
#for epiSign in [x for x in X.columns if x not in ('coord')]:
#    epiSize[epiSign] = sum(X[epiSign] > 0)  # Catch 100% methyl

# If detect NaN or null values -> stop here:
cols_with_na = {}
for a_col in X.columns:
    nb_na = sum(X[a_col].isna())
    if nb_na > 0:
        cols_with_na[a_col] = nb_na
assert not cols_with_na, f"Some cols have missing values: {cols_with_na}"

## Pre-processing

### Batch-effect correction using (py)combat

In [ ]:
# First we generate the list of batches:
dataset_251217 = ["HG002_combined","barcode04_combined"]
ref_sign = [ x for x in X.columns if x not in dataset_251217+['coord'] ]

batch = []
datasets = [ref_sign, dataset_251217]
for j in range(len(datasets)):
    batch.extend([j for _ in range(len(datasets[j]))])

# Then run (py)combat:
#X_corrected = pycombat(X, batch)

### Transpose + normalize

In [ ]:
USE_COMBAT = False
USE_NORMALIZED = False
EPISIGN_OF_INTEREST = ['HG002_combined','RMNS','Kleefstra','Kabuki','barcode04_combined']
# ONLY if 'full data from publi':
#EPISIGN_OF_INTEREST += ['EPI_01', 'EPI_02', 'EPI_08','EPI_19'] # 4 kab samples

# MirinFMF project
# Possible F samples:
#EPISIGN_OF_INTEREST = ["8ZZUUQM","CSG246672","CSG247928","CSG248017","CSG248018","CSG251485","CSG252095","CSG253228","CSG253525","CSG254002","CSG256433","CSG256435","E19WLVU","LQXGE7H","NL0500099","NL0800233","NL0800248","NL0800249","NL0800270","OD_2024101","PH5SJZD","PID_160383","PID_160392","PID_160401","PID_160403","PID_170302","PID_170308","PID_170365","PID_170366","R4XUJV2","R6CYPDM","RT8DFEP_FK2UZYZ","UAKAPLJ","V-87-DZ_16_10_n"]

# RNA:
EPISIGN_OF_INTEREST = ['SRR14863992','SRR14863859']

# Transpose (required):
X_t = X.T
if USE_COMBAT:
    X_t = X_corrected.T

# Remove 2nd row = size of epiSign then normalize
to_norm = X_t
if 'epiSize' in X_t.columns:
    to_norm = X_t.drop('epiSize', axis=1)
print(to_norm[to_norm.columns[0:3]].head())

# WARN: If norm, should be AFTER combat
to_PCA = to_norm
if USE_NORMALIZED:
    to_PCA = pd.DataFrame(StandardScaler().fit_transform(to_norm), columns=to_norm.columns, index=to_norm.index)

# Write corrected file:
for sample in ['barcode04_combined', 'HG002_combined']:
    #to_PCA.T[sample].to_csv(f"{sample}_corrected.tsv", sep="\t")
    print(f" > Wrote: {sample}_corrected.tsv")

## Vizu / dimension reduction

### PCA

In [ ]:
# Run PCA:
NB_COMPON=20
pca = PCA(
    n_components=20,
    random_state=42
)
pcs = pca.fit_transform(to_PCA.to_numpy())

print(f"Trustworthiness of the low-dimensional embedding for PCA: {trustworthiness(to_PCA, pcs, n_neighbors=2)}")
print( "Variance expliquée PC1 :", round( pca.explained_variance_ratio_[0] * 100, 2 ), "%" )
print( "Variance expliquée PC2 :", round( pca.explained_variance_ratio_[1] * 100, 2 ), "%" )
print( "Total variance explained:", sum(pca.explained_variance_ratio_)*100)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [ ]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[EPISIGN_OF_INTEREST])

In [ ]:
# Plot PCA
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df[[x_compon,y_compon]],
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

### UMAP-like

In [ ]:
# MEMO: Scikit-learn does NOT support UMAP
#       + Not planned to add it (2024): https://github.com/scikit-learn/scikit-learn/issues/19393
isomap = Isomap(n_components=2, n_neighbors=2).fit_transform(to_PCA)
# SpectralEmbedding:
#isomap = SpectralEmbedding(n_components=2).fit_transform(to_PCA)
print(f"Trustworthiness of the low-dimensional embedding for UMAP-like: {trustworthiness(to_PCA, isomap, n_neighbors=2)}")

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
iso_df = pd.DataFrame(
    isomap,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_iso = iso_df.loc[EPISIGN_OF_INTEREST]
print(subset_iso)

In [ ]:
fig_iso = plot_df(iso_df, EPISIGN_OF_INTEREST)
fig_iso.show()

print("Isomap and SpectralEmbedding gives shit results")

### UMAP (true)

In [ ]:
# MEMo: Joris uses 'n_neighbors=2'
res_umap = umap.UMAP(n_neighbors=2).fit_transform(to_PCA)
print(f"Trustworthiness of the low-dimensional embedding for UMAP: {trustworthiness(to_PCA, res_umap, n_neighbors=2)}")

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
umap_df = pd.DataFrame(
    res_umap,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_umap = umap_df.loc[EPISIGN_OF_INTEREST]
print(subset_umap)

In [ ]:
# Plot UMAP:
plot_df(umap_df, EPISIGN_OF_INTEREST)

### Heatmaps

# Simple one
# MEMO: Have to drop coord, otherwise does not plot correctly (too many rows probably)
print(EPISIGN_OF_INTEREST)
fig = px.imshow(
    to_PCA.T[EPISIGN_OF_INTEREST].T.reset_index(drop=True),
    text_auto=True
)
fig.update_layout(
    yaxis=dict(tickfont_size=7)
)

# MEMO: With index, plot broken with 'px.imshow'
names = to_PCA.index
coord = to_PCA.columns

print(to_PCA.info())

go.Figure(data=go.Heatmap(
    z = to_PCA.to_numpy(),
    x = coord,
    y = names
))

# MEMO: Have to drop index, otherwise plot broken
# Width/height empirical bellow
# Correct param is 'tickfont' (and not 'textfont')
fig = px.imshow(
    to_PCA.to_numpy(),
    y = names,
    x = coord,
    width = 15000,
    height = 1000
)
fig.update_layout(
    autosize=False,
    xaxis=dict(tickfont=dict(size=5)),
    yaxis=dict(tickfont=dict(size=5))
)

### t-SNE

In [ ]:
# Run t-SNE:
# MEMOs:
# - In Joris' paper they use 'preplex=2'
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=2).fit_transform(to_PCA)
print(f"Trustworthiness of the low-dimensional embedding for t-SNE: {trustworthiness(to_PCA, tsne, n_neighbors=2)}")

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_tsne = tsne_df.loc[EPISIGN_OF_INTEREST]
print(subset_tsne)

In [ ]:
# Plot t-SNE
fig_tsne = plot_df(tsne_df, EPISIGN_OF_INTEREST)
fig_tsne.show()

## Clustering

### K-means

In [ ]:
# Select optimal number of cluster using 'silouhette' method
sil_scores = {}

for k in range(2, 10):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = km.fit_predict(pcs)

    sil = silhouette_score(
        pcs,
        labels
    )

    sil_scores[k] = sil


best_k = max(
    sil_scores,
    key=sil_scores.get
)

print(
    "\nNombre optimal de clusters :",
    best_k
)

print(
    "Score silhouette :",
    sil_scores[best_k]
)

In [ ]:
# Clustering final avec le 'best nb cluster':
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50
)

clusters_kmeans = kmeans.fit_predict(pcs)

clusters_kmeans_df = pd.DataFrame(
    {
        "sample": X.T.index,
        "cluster": clusters_kmeans + 1
    }
)

print(clusters_kmeans_df.groupby('cluster').count())
#clusters_kmeans_df.to_csv('clusters_kmeans.tsv', sep="\t", index=False)

### h-Clust

In [ ]:
# Seaborn's clustermap does h-clust + visu:
color_interest_sample = lambda s: 'pink' if s in EPISIGN_OF_INTEREST else 'blue'

sns.set(font_scale=0.7)
hclust = sns.clustermap(
    to_PCA,
    metric='correlation',
    method='average',
    figsize=(20, 20),
    row_colors=list(map(color_interest_sample,to_PCA.index)),
)
print(EPISIGN_OF_INTEREST)

In [ ]:
# Recup des clusters du h-clust
from scipy.cluster.hierarchy import fcluster

# Les ech sont dans les lignes:
linkage = hclust.dendrogram_row.linkage

NB_CLUST = best_k
clusters_hclust = fcluster(
    linkage,
    t=NB_CLUST,
    criterion="maxclust"
)

# ALT: Decouper selon une dist plutot qu'un nombre de clust
fcluster(
    linkage,
    t=0.7,
    criterion="distance"
)

clusters_hclust_df = pd.DataFrame(
    {
        "sample": X.T.index,
        "cluster": clusters_hclust
    }
)
print(clusters_hclust_df.groupby('cluster').count())
#clusters_hclust_df.to_csv('clusters_hclust.tsv', sep="\t", index=False)

### hClust on PCA res (with method=ward)

In [ ]:
# Seaborn's clustermap does h-clust + visu:
hclust_pca = sns.clustermap(
    pcs_df,
    method='ward',
    figsize=(20, 20),
    row_colors=list(map(color_interest_sample,to_PCA.index)),
)

# Recup des clusters du h-clust
# Les ech sont dans les lignes:
linkage_pca = hclust_pca.dendrogram_row.linkage

NB_CLUST = best_k
clusters_hclust_pca = fcluster(
    linkage_pca,
    t=NB_CLUST,
    criterion="maxclust"
)

clusters_hclust_pca_df = pd.DataFrame(
    {
        "sample": X.T.index,
        "cluster": clusters_hclust_pca
    }
)
print(clusters_hclust_pca_df.groupby('cluster').count())
#clusters_hclust_pca_df.to_csv('clusters_hclust_pca.tsv', sep="\t", index=False)

### Bilan clustering

In [ ]:
# Compare predicted clusters between methods with ARI
# * 1.0 => clustering identique
# * 0.8+ => très forte concordance
# * 0.5 => concordance modérée
# * 0 => accord aléatoire
# * <0 => pire que le hasard

from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(
    clusters_kmeans,
    clusters_hclust_pca
)

print(ari)

In [ ]:
# Plot PCA, but colored by clusters

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=clusters_kmeans,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

## Classif

In [ ]:
from sklearn.metrics import accuracy_score

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.model_selection import cross_val_score

### Split train/test

In [ ]:
# For now use clusters predicted by k-means
y = clusters_kmeans

X_train, X_test, y_train, y_test = train_test_split(
    to_PCA,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Gene selection: 200 a 1000 suffisent
BEST_GENES = 200

### Random forest

In [ ]:
# Avantages :
# - peu sensible aux paramètres ;
# - capte les interactions non linéaires ;
# - fournit une importance des gènes.

pipe_rf = Pipeline([
    ("select", SelectKBest(f_classif, k=BEST_GENES)),
    ("rf", RandomForestClassifier(n_estimators=1000, random_state=42))
])
pipe_rf.fit(
    X_train,
    y_train
)

# Predict on 'test':
pred_rf = pipe_rf.predict(X_test)

print(
    accuracy_score(
        y_test,
        pred_rf
    )
)

# FIXME: Genes predictifs:
features_rf = pipe_rf.named_steps["rf"].feature_importances_

### SVM

In [ ]:
# Bonne option si nombre de gènes >> nombre d'échantillons (eg: 5000 genes vs 50 ech)

pipe_svm = Pipeline([
    ("select", SelectKBest(f_classif, k=BEST_GENES)),
    ("svm", SVC(kernel="linear", C=1))
])
pipe_svm.fit(
    X_train,
    y_train
)

# Predict on 'test':
pred_svm = pipe_svm.predict(X_test)

print(
    accuracy_score(
        y_test,
        pred_svm
    )
)

# FIXME: Genes predictifs:
features_svm = pipe_svm.named_steps["svm"].coef_

### Validation croisee

In [ ]:
# WARN: Uniquement sur le train

# SVM
scores_svm = cross_val_score(
    pipe_svm,
    X_train,
    y_train,
    cv=5
)
print(scores_svm)
print(scores_svm.mean())

# RF
scores_rf = cross_val_score(
    pipe_rf,
    X_train,
    y_train,
    cv=5
)
print(scores_rf)
print(scores_rf.mean())

# ALT: Use 'cv=StratifiedKFold'

### Add last row otherwise something show in above plot